# Round mosaics — live quick look at every round

Builds a mosaic (one per real imaging color) for whichever rounds you select
(`SELECTED_ROUNDS`), from a single frame per FOV near `TARGET_Z_UM` -- no
full z-stack read, so it's light enough to run continuously alongside a real
acquisition (reading straight off the NAS while HAL/Dave is still writing).
Flat-field correction is OFF by default (`ENABLE_FFC`, section 5) for speed.
By default, the 488 nm bead/focus-lock reference channel is excluded from
every round (`EXCLUDED_COLORS`, section 2) -- it's always at a fixed bead z
regardless of `TARGET_Z_UM`, not real tissue signal. This is a quick-look
tool, not a replacement for `analysis/02_round_scheduler.ipynb`'s production
mosaics (mid-z, optional FFC, built only once a round is 100% done) -- both
can run at the same time without conflicting; this notebook saves to
`SAMPLE_DIR/figures/`, not `analysis/mosaics/`.

**One state table, one loop, three use cases.** Section 6 builds a small
table -- one row per selected round, with how many of its FOVs are imaged
vs. already have a computed mosaic thumbnail -- and section 7 processes
whatever's outstanding, round by round, in the order the table lists them:
- `SELECTED_ROUNDS = "cells"`-only (or any single round) + `LIVE_LOOP = False`
  -- build it once from whatever's imaged right now, then stop. The old
  "on-demand specific round" mode.
- `SELECTED_ROUNDS = "all"` + `LIVE_LOOP = False` -- one pass over every
  round that's already fully imaged but not yet processed. The old
  "catch-up pass" mode.
- `SELECTED_ROUNDS = "all"` + `LIVE_LOOP = True` (the default) -- keeps
  re-checking and re-processing until every selected round is both fully
  imaged AND fully processed, picking up new rounds as imaging progresses.
  The old "live loop" mode -- meant to be started once and left running for
  the whole experiment.

Mix and match freely -- e.g. `SELECTED_ROUNDS = ["cells", 6]` with
`LIVE_LOOP = True` watches just those two rounds and stops once both are
done, ignoring everything else.

**Sequential only, by design, at least for now.** Parallelizing this across
a SLURM cluster (one array job per FOV, mirroring
`07_cluster_submit_analysis.ipynb`) was considered and deliberately deferred
-- that pattern exists in this repo for a much heavier per-FOV task (a full
multi-frame z-stack read, budgeted at 2 hours per SLURM task); this
notebook's per-FOV cost is a single named frame plus a cheap thumbnail, and
SLURM's own job-submission/queue overhead would plausibly dominate a task
that light rather than speed it up. Revisit this (or a local
`ProcessPoolExecutor`, matching `FOVScheduler`'s own pattern, which avoids
SLURM's overhead entirely) only if a real bulk backlog turns out too slow
sequentially.

**Reading from the NAS while it's being written to**: this notebook's reads
and HAL's writes share the same underlying disk/network link, so there's a
real, if usually small, risk of one slowing the other down -- see section
5's `CATCHUP_READ_DELAY_SEC` for the rationale behind its default.

## 1 — Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from scipy.spatial import KDTree

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import read_image_frames
from MERci.acquisition.configs    import find_frame_table_for_hal_config
from MERci.acquisition.positions  import find_exterior_fovs
from MERci.analysis.fov           import create_thumbnail
from MERci.analysis.round         import create_mosaic, create_mosaic_ffc
from MERci.analysis.ffc           import (
    compute_ffc_field_for_color, apply_ffc, save_ffc_field, load_ffc_field,
)
from MERci.scheduler              import resolve_round_flip_y

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "round_mosaics"   # used to namespace this notebook's cache + figure files

# Real stage z (um) to build every round's mosaic at -- for each round/color,
# the frame whose own z is closest to this gets used. One frame per FOV per
# color, no z-stack -- deliberately light enough to run continuously during a
# real acquisition. Change and re-run section 3 to pick a different depth
# (e.g. to match where your tissue actually has signal).
TARGET_Z_UM = 10.0

# Colors to leave out of every round's resolution (section 3) -- by default
# just the 488 nm bead/reference channel, imaged only for HAL's own focus
# lock at a fixed bead z regardless of TARGET_Z_UM (not real tissue signal,
# and always far from TARGET_Z_UM -- the actual source of a ">5um away"
# warning otherwise firing on nearly every round). Set to [] to include
# every real color again, e.g. to build a bead-channel mosaic for a specific
# round (combine with SELECTED_ROUNDS below rather than watching/catching
# up on beads for every round).
EXCLUDED_COLORS = [488.0]

# Which round(s) to build/watch mosaics for (section 6/7) -- "all" (default)
# considers every round with at least one resolved color; otherwise a list
# mixing round labels ("cells", resolved by imaging_type -- same convention
# as correct_camera_rotation.ipynb's own ROUND_IMAGING_TYPE) and/or explicit
# imaging_round numbers, e.g. ["cells", 6]. Processed in the order given
# (or ascending round id, for "all").
SELECTED_ROUNDS = "all"

# True (default): keep re-checking/re-processing until every selected round
# is both fully imaged AND fully processed (picking up newly-imaged FOVs and
# newly-started rounds as they appear) -- meant to be started once and left
# running for the whole experiment. False: process whatever's available
# right now, once, then stop -- e.g. for a quick one-off look at a single
# round (SELECTED_ROUNDS="cells", LIVE_LOOP=False), or one catch-up pass
# over everything already finished (SELECTED_ROUNDS="all", LIVE_LOOP=False).
LIVE_LOOP = True

# Flat-field correction -- off by default (fastest option, no extra reads).
# Turning it on costs FFC_N_FOVS extra one-time reads PER COLOR (not per
# round -- vignetting is a fixed optical property, cached to disk after the
# first computation) -- see section 5.
ENABLE_FFC = False
FFC_N_FOVS = 10

POLL_INTERVAL_SEC = 5         # must be well under the time to acquire one FOV
MAX_RUNTIME_MIN    = 24*60*7  # safety cap -- this notebook is meant to run for a
                               # whole multi-round experiment, not just one round

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

THUMBNAILS_DIR = config.analysis_dir / "thumbnails"   # shared with 01_fov_scheduler.ipynb's own convention
THUMBNAILS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME   # FFC fields only (NOTEBOOK_GUIDELINES.md #2)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# Deliberately SAMPLE_DIR/figures/, not analysis/mosaics/ -- see markdown
# above: this is a quick-look tool, kept out of the way of the production
# mosaics analysis/02_round_scheduler.ipynb builds at the same filenames.
FIGURES_DIR = SAMPLE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"FOVs (total) : {meta.n_fovs}")
print(f"Rounds       : {sorted(meta.rounds)}")
print(f"TARGET_Z_UM  : {TARGET_Z_UM}")
print(f"ENABLE_FFC   : {ENABLE_FFC}")
print(f"LIVE_LOOP    : {LIVE_LOOP}")

## 3 — Resolve each round's real colors -> nearest-z frame index

Reads each round's own frame table (via its HAL config, same resolution
`prepare_imaging` already uses) and picks, per real color (excluding
`EXCLUDED_COLORS`, section 2), the frame whose actual `z` (um) is closest to
`TARGET_Z_UM`. Rounds sharing the same HAL config (common for repeated bits
rounds) resolve identically, but each round is still looked up independently
since nothing guarantees that in general.

In [ ]:
def resolve_round_color_frames(round_id):
    # {color_nm: frame_idx} for round_id's own frame table -- the frame
    # closest to TARGET_Z_UM for every real color it has, except
    # EXCLUDED_COLORS (and blanks, which have no color at all).
    color_frames = {}
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        frame_table_path = find_frame_table_for_hal_config(
            config.settings_dir / s.hal_config, config.metadata_dir)
        if frame_table_path is None:
            continue
        frame_table = pd.read_csv(frame_table_path)
        for color in sorted(frame_table["color"].dropna().unique()):
            if any(round(color) == round(excluded) for excluded in EXCLUDED_COLORS):
                continue
            candidates = frame_table[frame_table["color"].round(0) == round(color)]
            frame_idx = int((candidates["z"] - TARGET_Z_UM).abs().idxmin())
            resolved_z = float(candidates.loc[frame_idx, "z"])
            color_frames[float(color)] = frame_idx
            if abs(resolved_z - TARGET_Z_UM) > 5.0:
                print(f"  round {round_id}, color {color:.0f} nm: nearest available z is "
                      f"{resolved_z:.1f} um (requested {TARGET_Z_UM:.1f} um) -- frame {frame_idx}")
    return color_frames


ROUND_COLOR_FRAMES = {}
for round_id in sorted(meta.rounds):
    cf = resolve_round_color_frames(round_id)
    if cf:
        ROUND_COLOR_FRAMES[round_id] = cf
    print(f"Round {round_id}: colors {sorted(cf)} -> frame indices {cf}")

## 4 — Resolve round labels and `SELECTED_ROUNDS`

A round's label is `"cells"` if one of its series has `imaging_type=="cells"`,
else its raw `imaging_round` number -- matching how `SELECTED_ROUNDS`
entries are themselves resolved (a string by `imaging_type`, an int
directly).

In [ ]:
def resolve_round_by_imaging_type(imaging_type):
    target = imaging_type.strip().lower()
    for round_id in sorted(meta.rounds):
        for s in meta.series_for_round(round_id):
            if (s.imaging_type or "").strip().lower() == target:
                return round_id
    return None


def resolve_round_token(token):
    # int -> that round id directly; str -> resolved by imaging_type.
    if isinstance(token, str):
        round_id = resolve_round_by_imaging_type(token)
        if round_id is None:
            raise ValueError(f"No round has a series with imaging_type={token!r} "
                              f"-- check round_info.csv, or use an explicit round id.")
        return round_id
    return int(token)


def round_label_for(round_id):
    for s in meta.series_for_round(round_id):
        if (s.imaging_type or "").strip().lower() == "cells":
            return "cells"
    return round_id


if SELECTED_ROUNDS == "all":
    SELECTED_ROUND_IDS = sorted(ROUND_COLOR_FRAMES)
else:
    SELECTED_ROUND_IDS = [resolve_round_token(t) for t in SELECTED_ROUNDS]
    unresolved = [r for r in SELECTED_ROUND_IDS if r not in ROUND_COLOR_FRAMES]
    if unresolved:
        raise ValueError(f"Round id(s) {unresolved} have no resolved colors "
                          f"(see section 3) -- check round_info.csv.")

print(f"SELECTED_ROUND_IDS: {SELECTED_ROUND_IDS} "
      f"(labels: {[round_label_for(r) for r in SELECTED_ROUND_IDS]})")

## 5 — Shared helpers: read/thumbnail/build one round's mosaic

Used by sections 6 and 7 below -- kept in one place, run before either of
them, so it doesn't matter which one you run first (an earlier version of
this notebook defined these inside one specific mode's own section, which
broke another mode with a `NameError` if it ran first).

**Flat-field correction, if `ENABLE_FFC`** (section 2, off by default):
samples come from real exterior FOVs (the outer edge of the imaged grid, via
`find_exterior_fovs` -- the same helper the production FFC pipeline uses)
that are ALREADY imaged in whatever round first needs that color's field --
not a fixed reference round, since the first-processed round might not be
the first one that actually has all `FFC_N_FOVS` (default 10) exterior FOVs
done yet. Cached to `analysis/cache/round_mosaics/ffc_field_{color}nm.npz`
and reused for every round afterward, since vignetting is a fixed property
of the microscope/channel, not of any one round. **Known tradeoff**: a round
processed while it still has unimaged FOVs remaining (the live case) gets
FFC divided out frame-by-frame right after reading, then thumbnailed/
stretched independently -- keeping a whole in-progress round's raw frames
in memory isn't affordable. A round that's already 100% imaged when it's
first processed gets the full production treatment instead (real raw
frames, ONE shared contrast stretch across the whole canvas via
`create_mosaic_ffc`), which looks visibly better. Either way, re-processing
a round later (e.g. after it finishes) upgrades it automatically next time
this notebook decides that round still needs work -- unlike the earlier
three-separate-modes version, there's no separate "already built, skip" file
check blocking a quality upgrade; `processed_fovs` in section 6/7's state
table only reflects thumbnail completeness, not mosaic-canvas quality.

**`CATCHUP_READ_DELAY_SEC`** -- reading many not-yet-thumbnailed FOVs back
to back is the only SUSTAINED read burst this notebook does; once a round is
fully processed, later cycles just reuse cached thumbnails (no sleep, no new
reads) until new FOVs appear. This burst competes with an ACTIVE round's
writes for the same disk/network link at the exact moment it happens. There
is no universal safe number here (it depends on this NAS's real hardware/
network capacity and how much other traffic already uses it, which nothing
in this notebook can measure for you) -- the default below is a
conservative, deliberately small pacing gap chosen for a different,
checkable reason: HAL's own write demand is small and steady (one frame is
`image_size_px**2 * 2` bytes -- e.g. ~8 MB for a 2048x2048 uint16 frame --
written roughly once per `exposure_time`, typically a few hundred ms, i.e.
on the order of tens of MB/s), well under what even a modest 1 GbE link can
carry (~118 MB/s) on bytes alone -- so the real risk isn't raw throughput,
it's IOPS/seek contention from many small, scattered file reads landing on
the same storage the sequential HAL write stream is using, which
byte-counting alone doesn't capture. `CATCHUP_READ_DELAY_SEC=0.02` (20 ms)
keeps this notebook's own FRESH-read rate capped at ~50 files/sec regardless
of how fast the NAS could otherwise serve them, which is slow enough to
leave real headroom without making a full round's mosaic impractically slow
(about 20s of pure pacing per 1000-FOV round, one-time). Treat this as a
starting point, not a proof of safety -- if you suspect it's still affecting
acquisition, check whether HAL's own per-frame write timing visibly changes
while this runs (real file-write-mtime gaps, the same technique
`misc/measure_tissue_thickness_test.ipynb` section 8 already uses) and
raise the delay if so.

In [ ]:
coords_arr   = np.array([meta.fovs[f].position for f in sorted(meta.fovs)])
nn_dist, _   = KDTree(coords_arr).query(coords_arr, k=2)
STEP_SIZE_UM = float(np.median(nn_dist[:, 1]))
EXTERIOR_FOV_IDS = find_exterior_fovs(
    {f: meta.fovs[f].position for f in meta.fovs}, STEP_SIZE_UM)

_ffc_fields = {}   # {color_nm: np.ndarray} -- in-memory cache for this session


def get_or_compute_ffc_field(color_nm, frame_idx, available_fov_ids, series):
    if not ENABLE_FFC:
        return None
    if color_nm in _ffc_fields:
        return _ffc_fields[color_nm]

    cache_path = CACHE_DIR / f"ffc_field_{color_nm:.0f}nm.npz"
    if cache_path.exists():
        field, _ = load_ffc_field(cache_path)
        _ffc_fields[color_nm] = field
        print(f"Loaded cached FFC field ({color_nm:.0f} nm): {cache_path}")
        return field

    candidate_ids = sorted(EXTERIOR_FOV_IDS & set(available_fov_ids))[:FFC_N_FOVS]
    if not candidate_ids:
        print(f"  FFC ({color_nm:.0f} nm): no exterior FOVs available yet -- will retry later.")
        return None

    samples = []
    for fov_id in candidate_ids:
        paths = [s.resolve_path(fov_id, config.image_suffix) for s in series]
        existing = [p for p in paths if p.exists()]
        if existing:
            samples.append((existing[0], frame_idx))
    if not samples:
        return None

    field, field_meta = compute_ffc_field_for_color(
        samples, frame_width=config.frame_width, frame_height=config.frame_height)
    save_ffc_field(cache_path, field, field_meta)
    _ffc_fields[color_nm] = field
    print(f"Computed FFC field ({color_nm:.0f} nm) from {len(samples)} exterior FOV(s): {cache_path}")
    return field


print(f"STEP_SIZE_UM       : {STEP_SIZE_UM:.1f} um")
print(f"Exterior FOV count : {len(EXTERIOR_FOV_IDS)}")


def round_imaged_fov_ids(round_id):
    series = meta.series_for_round(round_id)
    return [
        fov_id for fov_id in sorted(meta.fovs)
        if any(s.resolve_path(fov_id, config.image_suffix).exists() for s in series)
    ]


def thumbnail_path_for(image_path, frame_idx):
    return THUMBNAILS_DIR / f"{image_path.stem}_frame{frame_idx:03d}.png"


def fov_is_processed(round_id, fov_id, color_frames, series):
    # True iff fov_id already has a cached thumbnail for EVERY color of this
    # round -- a round only counts as "processed" once ALL its colors are
    # done, not just the first one.
    existing = [s.resolve_path(fov_id, config.image_suffix) for s in series]
    existing = [p for p in existing if p.exists()]
    if not existing:
        return False
    image_path = existing[0]
    return all(thumbnail_path_for(image_path, frame_idx).exists()
               for frame_idx in color_frames.values())


def mosaic_path(round_id, color_nm):
    return FIGURES_DIR / f"{NOTEBOOK_NAME}.round{round_id:03d}_{color_nm:.0f}nm.png"


def build_round_mosaic(round_id, color_frames, fov_ids):
    # Read one frame per fov_id per color (paced by CATCHUP_READ_DELAY_SEC,
    # but only for FRESH reads -- an already-cached thumbnail costs no sleep),
    # and save/return one mosaic per color -- FFC-corrected (whole-canvas
    # shared contrast stretch) if ENABLE_FFC, else independently-stretched
    # thumbnails (cached to THUMBNAILS_DIR, shared with 01_fov_scheduler.
    # ipynb's own convention). Returns {color_nm: canvas}.
    series  = meta.series_for_round(round_id)
    flip_y  = resolve_round_flip_y(round_id, config, meta)
    canvases = {}
    for color_nm, frame_idx in color_frames.items():
        ffc_field = get_or_compute_ffc_field(color_nm, frame_idx, fov_ids, series)
        thumbnails, raw_frames, positions = {}, {}, {}
        for fov_id in fov_ids:
            existing = [s.resolve_path(fov_id, config.image_suffix) for s in series]
            existing = [p for p in existing if p.exists()]
            if not existing:
                continue
            image_path = existing[0]
            if ffc_field is not None:
                raw_frames[fov_id] = read_image_frames(
                    image_path, [frame_idx],
                    frame_width=config.frame_width, frame_height=config.frame_height,
                )[0]
                time.sleep(CATCHUP_READ_DELAY_SEC)
                # Also write the plain (non-FFC) thumbnail as a "this FOV has
                # been processed" marker -- fov_is_processed checks THIS file,
                # not the FFC canvas, so without it an FFC-enabled round could
                # never register as processed and LIVE_LOOP=True would loop on
                # it forever. Same raw-frame convention as 01_fov_scheduler.
                # ipynb's own thumbnails (not FFC-divided), so the cache stays
                # directly reusable if ENABLE_FFC is later turned off.
                thumb_path = thumbnail_path_for(image_path, frame_idx)
                if not thumb_path.exists():
                    create_thumbnail(raw_frames[fov_id], thumb_path, target_size=config.thumbnail_size,
                                      percentile_clip=config.thumbnail_percentile_clip)
            else:
                thumb_path = thumbnail_path_for(image_path, frame_idx)
                if thumb_path.exists():
                    from PIL import Image
                    thumb = np.array(Image.open(str(thumb_path)))
                else:
                    frame = read_image_frames(
                        image_path, [frame_idx],
                        frame_width=config.frame_width, frame_height=config.frame_height,
                    )[0]
                    thumb = create_thumbnail(frame, thumb_path, target_size=config.thumbnail_size,
                                              percentile_clip=config.thumbnail_percentile_clip)
                    time.sleep(CATCHUP_READ_DELAY_SEC)
                thumbnails[fov_id] = thumb
            positions[fov_id] = meta.fovs[fov_id].position

        if ffc_field is not None and raw_frames:
            canvases[color_nm] = create_mosaic_ffc(
                raw_frames, positions, mosaic_path(round_id, color_nm),
                ffc_field=ffc_field, crop_px=0, thumbnail_size=config.thumbnail_size,
                padding=config.mosaic_padding, flip_y=flip_y,
                percentile_clip=config.thumbnail_percentile_clip,
            )
        elif thumbnails:
            canvases[color_nm] = create_mosaic(
                thumbnails, positions, mosaic_path(round_id, color_nm),
                thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding, flip_y=flip_y,
            )
    return canvases


CATCHUP_READ_DELAY_SEC = 0.02   # see markdown above for the rationale

## 6 — Build the state table

One row per selected round: how many of its FOVs are imaged right now, how
many already have a computed thumbnail for every one of that round's colors
("processed"), and the experiment's total FOV count. Re-run this cell any
time for a fresh snapshot -- section 7 also rebuilds it itself on every
poll, so this is mainly for a quick look without starting the full loop.

In [ ]:
def build_state_df(round_ids):
    rows = []
    for round_id in round_ids:
        color_frames = ROUND_COLOR_FRAMES[round_id]
        series = meta.series_for_round(round_id)
        imaged_ids = round_imaged_fov_ids(round_id)
        processed_ids = [f for f in imaged_ids if fov_is_processed(round_id, f, color_frames, series)]
        rows.append({
            "round": round_label_for(round_id), "round_id": round_id,
            "imaged_fovs": len(imaged_ids), "processed_fovs": len(processed_ids),
            "total_fovs": meta.n_fovs,
        })
    return pd.DataFrame(rows, columns=["round", "round_id", "imaged_fovs", "processed_fovs", "total_fovs"])


STATE_DF = build_state_df(SELECTED_ROUND_IDS)
display(STATE_DF)

## 7 — Process selected rounds (one loop for all three use cases)

Rebuilds the state table every cycle and processes any round with
`processed_fovs < imaged_fovs`, in the order `SELECTED_ROUND_IDS` lists them
-- reading/thumbnailing just the FOVs that are imaged but not yet processed,
then redrawing that round's mosaic(s). If `LIVE_LOOP`, keeps looping until
every selected round is both fully IMAGED and fully PROCESSED (or
`MAX_RUNTIME_MIN` is exceeded); otherwise processes whatever's available
once and stops. Interrupt the kernel to stop early at any time -- whatever's
been built so far is already saved to `figures/`.

In [ ]:
def show_round_mosaic(round_id, canvases, label):
    colors = sorted(canvases)
    fig, axes = plt.subplots(1, max(len(colors), 1), figsize=(6 * max(len(colors), 1), 6), squeeze=False)
    for ax, color_nm in zip(axes[0], colors):
        ax.imshow(canvases[color_nm], cmap="gray")
        ax.set_title(f"round {label} — {color_nm:.0f} nm")
        ax.axis("off")
    fig.tight_layout()
    clear_output(wait=True)
    display(fig)
    plt.close(fig)


start_time = time.time()
poll_count = 0

try:
    while True:
        if (time.time() - start_time) > MAX_RUNTIME_MIN * 60:
            print(f"Stopping: MAX_RUNTIME_MIN={MAX_RUNTIME_MIN} exceeded.")
            break

        poll_count += 1
        state_df = build_state_df(SELECTED_ROUND_IDS)
        fully_done = True

        for _, row in state_df.iterrows():
            round_id = row["round_id"]
            if row["imaged_fovs"] < row["total_fovs"]:
                fully_done = False   # still being imaged -- keep watching even with nothing to process yet
            if row["processed_fovs"] < row["imaged_fovs"]:
                fully_done = False
                color_frames = ROUND_COLOR_FRAMES[round_id]
                fov_ids = round_imaged_fov_ids(round_id)
                print(f"poll #{poll_count} | round {row['round']}: processing "
                      f"{row['processed_fovs']} -> {len(fov_ids)}/{row['total_fovs']} FOVs...")
                canvases = build_round_mosaic(round_id, color_frames, fov_ids)
                show_round_mosaic(round_id, canvases, row["round"])

        # Refresh before printing -- state_df above was captured at the TOP
        # of this cycle, so it would otherwise still show pre-processing
        # counts for whatever this cycle just built, right below the mosaic
        # image proving it was actually built.
        state_df = build_state_df(SELECTED_ROUND_IDS)
        elapsed = time.time() - start_time
        print(f"poll #{poll_count} | elapsed {elapsed / 60:.1f} min | next check in {POLL_INTERVAL_SEC}s")
        print(state_df.to_string(index=False))

        if not LIVE_LOOP:
            print("LIVE_LOOP is False -- one pass complete.")
            break
        if fully_done:
            print("All selected rounds fully imaged and processed.")
            break

        time.sleep(POLL_INTERVAL_SEC)
except KeyboardInterrupt:
    print(f"Stopped by user after poll #{poll_count}.")